# VCC 2026 — un solo notebook, più sorgenti

Orchestrazione Colab: **preflight → selezione → acquisizione → QC → deriva → export**.
Checkpoint indipendenti per blocco. Un solo file originale grande alla volta.

**Il run `catalog_2026-09-15T143641Z` non è un fallimento.** `FETCH_BLOCKS` era vuoto, quindi `plan_only=True` e nessun download. Questa copia, dopo il preflight, sceglie da sola il prossimo blocco che ci sta.

**Pilot Colab già fatto** (`/content/vcc-persist/remote_ingest_2026-09-15T140308Z`): HepG2 parità ok, resume 200→completo, Jiang non scaricato. Disco dopo il pilot **87,25 GiB**. `/content/vcc-persist` è il disco della VM: **non persiste**. RAM ~11 GB: rimisurarla sotto. Quel pilot **non** prova l'ingestione da 65,8 GiB né la ripresa del training.

Il file «61,3 GB» del profilo è `K562_gwps_raw_singlecell_01.h5ad`: **65.830.941.948 byte**, md5 `887e3e6a…`, figshare 35775507. Non è lo specchio scPerturb da 8,8 GB.

Monta Drive nella prima cella. Senza Drive il catalogo può rifare HepG2 (~0,85 GB) ma **non** parte il file da 65,8 GiB (morirebbe con la VM).

Aprire: File → Carica notebook (`vcc2026-colab.ipynb` sul Desktop), prima cella → `vcc2026-colab.zip` **nuovo** (lo zip vecchio non ha `recommend_fetch_ids`).

In [ ]:
import os, sys, tarfile, zipfile, shutil
from pathlib import Path

# None = auto after preflight. [] = plan only. list = explicit ids.
FETCH_BLOCKS = None
SELECT_BLOCKS = None  # None = catalog default_order
DRY_RUN = False
MOUNT_DRIVE = True
ALLOW_EPHEMERAL_LARGE = False
IN_COLAB = 'google.colab' in sys.modules

def _find(patterns):
    roots = [Path.cwd(), Path('/content'), Path('/kaggle/input'), Path('/kaggle/working')]
    hits = []
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            hits.extend(root.glob(pat))
            hits.extend(root.glob('**/' + pat))
    return [p for p in hits if p.is_file()]

if IN_COLAB:
    os.environ['VCC2026_REMOTE'] = '1'
    if MOUNT_DRIVE:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive')
        print('Drive mounted', Path('/content/drive/MyDrive').is_dir())
    from google.colab import files
    already = _find(['vcc2026-colab.zip', 'source_snapshot.tar.gz'])
    if not already:
        print('Scegli file: vcc2026-colab.zip dal Desktop')
        files.upload()
    for z in _find(['vcc2026-colab.zip', '*.zip']):
        print('unzip', z)
        with zipfile.ZipFile(z) as zh:
            zh.extractall('/content/vcc-bundle')
        break

snap = next(iter(_find(['source_snapshot.tar.gz'])), None)
if (Path.cwd() / 'src' / 'vcc2026').is_dir():
    REPO = Path.cwd()
elif snap is not None:
    REPO = Path('/content/vcc2026-src') if IN_COLAB else Path.cwd() / 'vcc2026-src'
    REPO.mkdir(parents=True, exist_ok=True)
    with tarfile.open(snap, 'r:gz') as tar:
        tar.extractall(REPO)
else:
    raise SystemExit('Manca source_snapshot.tar.gz (nello zip vcc2026-colab.zip).')

os.environ['VCC2026_REPO'] = str(REPO.resolve())
sys.path.insert(0, str(REPO / 'src'))
try:
    from vcc2026.remote_catalog import recommend_fetch_ids  # noqa: F401
except ImportError:
    raise SystemExit(
        'Zip vecchio in questa VM: cancella /content/vcc-bundle e '
        '/content/vcc2026-colab.zip, poi carica il vcc2026-colab.zip nuovo dal Desktop.'
    )
print('repo', REPO.resolve())
print('DRY_RUN', DRY_RUN, 'FETCH_BLOCKS', FETCH_BLOCKS)

## 1. Preflight

Rimisura Python, RAM, disco, persistenza. `/content/vcc-persist` è temporaneo.
Se Drive è montato, **matrici e report** vanno sotto `/content/drive/MyDrive/vcc2026/`.

In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'anndata', 'pyyaml'])

from vcc2026.remote_ingest import detect_runtime, resolve_paths
from vcc2026.remote_catalog import persist_kind
from vcc2026.resources import snapshot, GiB

print('runtime', detect_runtime())
paths = resolve_paths(repo=REPO)
print(paths.as_dict())
print('persist_kind', persist_kind(paths.persist))
if persist_kind(paths.persist)['kind'] == 'colab_vm':
    print('AVVISO: disco VM. HepG2 si può rifare; il file da 65,8 GiB NON parte finché non monti Drive.')

axis_dest = paths.data / 'raw' / 'controls' / 'gene_names.csv'
axis_dest.parent.mkdir(parents=True, exist_ok=True)
if not axis_dest.exists():
    found = next(iter(Path('/content').rglob('gene_names.csv') if Path('/content').exists() else []), None)
    found = found or next(iter(Path.cwd().rglob('gene_names.csv')), None)
    if found is None:
        raise SystemExit('gene_names.csv assente dallo zip')
    shutil.copy(found, axis_dest)
os.environ['VCC2026_DATA_ROOT'] = str(paths.data)

now = snapshot(paths.persist)
print('RAM total GiB', None if now.ram_total_bytes is None else now.ram_total_bytes / GiB)
print('RAM avail GiB', None if now.ram_available_bytes is None else now.ram_available_bytes / GiB)
print('disk free GiB', now.disk_free_bytes / GiB)
print('disk path', now.disk_path)
print('USER REPORT after HepG2 pilot: disk 87.25 GiB, RAM ~11 GiB — numbers above are this process.')

## 2. Selezione sorgenti

Catalogo in `configs/remote_catalog.yaml`. Byte e md5 da evidenza, non dal profilo.
HepG2 si salta se la dimensione coincide. Con `FETCH_BLOCKS is None` il prossimo fetch è automatico: HepG2 se manca, poi K562 GW se Drive è montato e il disco basta.

In [ ]:
from vcc2026.remote_catalog import dest_for, load_catalog, plan_block, recommend_fetch_ids
from vcc2026.resources import GiB, snapshot

catalog = load_catalog(REPO / 'configs' / 'remote_catalog.yaml')
meas = snapshot(paths.persist)
floor = 2.0 if paths.remote else 10.0
print('floor GiB', floor, '(proposed remote; laptop D-005 is 10)')
selected = catalog.selected(SELECT_BLOCKS)
complete_ids = set()
for block in selected:
    dest = dest_for(block, paths.data)
    if dest.exists() and block.bytes is not None and dest.stat().st_size == block.bytes:
        complete_ids.add(block.id)
pk = persist_kind(paths.persist)
rec = recommend_fetch_ids(
    catalog,
    free_bytes=meas.disk_free_bytes,
    ram_available_bytes=meas.ram_available_bytes,
    floor_bytes=int(floor * GiB),
    persist_survives_session=bool(pk['survives_session']),
    complete_ids=complete_ids,
    allow_ephemeral_large=ALLOW_EPHEMERAL_LARGE,
    select_ids=SELECT_BLOCKS,
)
print('complete_ids', sorted(complete_ids))
print('recommendation', rec['ids'])
for note in rec['notes']:
    print(' ', note)
for block in selected:
    want = block.id in rec['ids']
    plan = plan_block(
        block,
        free_bytes=meas.disk_free_bytes,
        ram_available_bytes=meas.ram_available_bytes,
        floor_bytes=int(floor * GiB),
        fetch=want and not DRY_RUN,
        already_complete=block.id in complete_ids,
    )
    adv = '' if not plan['advertised_bytes'] else f" advertised={plan['advertised_bytes']}"
    print(f"{block.id:28} {plan['decision']:14} bytes={plan['bytes']}{adv}  {plan['reason']}")

if FETCH_BLOCKS is None:
    if DRY_RUN or not paths.remote:
        FETCH_BLOCKS = []
        print('Auto-fetch withheld (DRY_RUN or local). Would have fetched:', rec['ids'])
    else:
        FETCH_BLOCKS = list(rec['ids'])
print('FETCH_BLOCKS', FETCH_BLOCKS)
if not FETCH_BLOCKS and not DRY_RUN and paths.remote:
    print('Niente da scaricare: monta Drive per il file da 65,8 GiB, oppure imposta FETCH_BLOCKS a mano.')

## 3–6. Acquisizione, QC, deriva, export

Un blocco per volta. Checkpoint in `out/checkpoints/<id>.json` (non si sovrascrivono se complete).
QC e maschera genica / NTC da `obs`/`var`, senza materializzare X. Jiang RDS: download possibile, conversione **bloccata** (serve R).
Firme log2FC complete = `scripts/53_build_hepg2_signatures.py`, non questa cella.

K562 GW è **65.830.941.948 byte**. Su Drive può durare ore; non chiudere il runtime.

In [ ]:
from vcc2026.remote_catalog import run_catalog
from datetime import datetime, timezone
from pathlib import Path
import os

stamp = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H%M%SZ')
out = paths.persist / f'catalog_{stamp}'
report = run_catalog(
    out=out,
    repo=Path(os.environ['VCC2026_REPO']),
    fetch_ids=list(FETCH_BLOCKS or []),
    select_ids=SELECT_BLOCKS,
    catalog_path=REPO / 'configs' / 'remote_catalog.yaml',
    plan_only=bool(DRY_RUN or not FETCH_BLOCKS),
    allow_ephemeral_large=ALLOW_EPHEMERAL_LARGE,
)
print('out', out)
print('persist_kind', report['persist_kind'])
print('plan_only', report['plan_only'])
print('fetch_ids', report['fetch_ids'])
print('not_a_cloud_proof', report['not_a_cloud_proof'])
for row in report['blocks']:
    p = row['plan']
    print(f"{p['block_id']:28} {row['status']:16} {p['reason']}")
if report['persist_kind'].get('survives_session'):
    print('Output su Drive (o disco duraturo):', out)
else:
    print('Scarica la cartella', out, 'prima che la VM Colab muoia.')